# Pre-final report: product-feature clustering

## Goal and data

We group feature titles that describe the same product property. The catalog contains 208 products, 445 unique English titles, and descriptions for 433 titles; 22 descriptions are unpaired. There are no ground-truth cluster labels, so the results below describe patterns in the current similarity matrix, not verified equivalence.

## Similarity construction

Each title pair receives `0.50 × semantic + 0.25 × TF-IDF + 0.25 × fuzzy` similarity. A 0.30 penalty is applied to 72 detected winter-versus-windproof pairs, and scores are clipped to `[0, 1]`. Clustering uses distance `D = 1 - similarity`.

Semantic embeddings help match paraphrases and synonyms that share few words. We blend the title vector with vectors for its aligned descriptions so the title's meaning is informed by product context.

TF-IDF/IDF gives more weight to informative, less common description terms than to generic terms. This supplies direct lexical evidence alongside the embeddings; titles without descriptions simply have no TF-IDF evidence.

RapidFuzz catches near-identical titles with spelling, punctuation, or word-order differences. It is fast and complements the other signals, but cannot by itself connect many paraphrases.

## Clustering methods

All methods start from the same 445 × 445 similarity matrix `S`. Agglomerative, DBSCAN, and HDBSCAN use its distance form `D = 1 - S`; spectral clustering uses `S` to build a weighted graph. They differ in the mathematical rule used to form groups.

### Agglomerative clustering

Start with one cluster per title. For each pair of clusters `A` and `B`, average linkage defines their distance as `d(A,B) = sum(D[i,j] for i in A, j in B) / (|A| × |B|)`. The algorithm repeatedly merges the pair with the smallest average distance, then recalculates distances to the new cluster. Here it stops merging at distance threshold `0.70`. Merges are irreversible, and every title remains assigned to a cluster.

### DBSCAN

For title `i`, its neighborhood is `Nε(i) = {j : D[i,j] ≤ ε}`. A title is a **core point** if this neighborhood contains at least `min_samples` titles (including itself); here `ε=0.40` and `min_samples=2`. DBSCAN grows clusters through chains of neighboring core points and can attach nearby border points. Titles not reachable from a core point are **noise**. This report additionally treats any resulting one-title cluster as noise.

### HDBSCAN

HDBSCAN estimates local density using each title's distance to its `min_samples`-th neighbor, called its core distance `cₖ(i)`. It replaces each pairwise distance with mutual reachability distance, `dₘᵣ(i,j) = max(cₖ(i), cₖ(j), D[i,j])`, then builds a minimum spanning tree and a hierarchy of groups across density levels. It condenses branches smaller than `min_cluster_size` and selects stable clusters using the default excess-of-mass rule; titles outside the selected groups are noise. This run uses `min_cluster_size=2` and `min_samples=2`, so groups as small as two titles are permitted.

### Spectral clustering

Spectral clustering first creates a weighted graph `W` from each title's 15 most similar neighbors, then symmetrizes the connections. It forms the normalized graph Laplacian `L = I - Δ⁻¹ᐟ² W Δ⁻¹ᐟ²`, where `Δ` contains each node's weighted degree. The eigenvectors for the `k` smallest eigenvalues embed titles in a lower-dimensional space; after row normalization, K-means partitions that space. Here an eigengap search over `k=20…80` selects `k=25`, and K-means assigns every title to one of 25 clusters.

The reference settings make the comparison reproducible and apply the same titles and matrix. No labeled examples are available to establish which clusters are semantically correct. The later code cells compute the scores and generate the results graph.


In [1]:
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.cluster import AgglomerativeClustering, DBSCAN, HDBSCAN, KMeans
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score

matrix_path = Path("data/similarity.npz")
if not matrix_path.exists():
    raise FileNotFoundError("Run this notebook from the repository root; data/similarity.npz was not found.")

with np.load(matrix_path) as saved:
    S = saved["similarity"].astype(float)
    attributes = saved["titles"].tolist()

n = len(attributes)
assert S.shape == (n, n), f"Expected ({n}, {n}), got {S.shape}"
assert np.allclose(S, S.T), "Similarity matrix must be symmetric"
assert np.isfinite(S).all() and S.min() >= 0 and S.max() <= 1 + 1e-9
D = np.clip(1 - S, 0, 1)
np.fill_diagonal(D, 0)
print(f"Loaded {n} titles; similarity range [{S.min():.3f}, {S.max():.3f}]; distance shape {D.shape}")

Loaded 445 titles; similarity range [0.000, 1.000]; distance shape (445, 445)


## Evaluation approach

The silhouette score summarizes how close each title is to its own cluster compared with the nearest alternative. For a title, `a` is its average distance to its own cluster and `b` is its smallest average distance to another cluster; its score is `(b - a) / max(a, b)`. The score ranges from -1 to 1: near 1 means compact and separated, around 0 means overlapping groups, and below 0 suggests the title may fit another group better.

As a rough guide—not a universal rule—scores above 0.7 often indicate strong separation, 0.5–0.7 reasonable separation, 0.25–0.5 weak separation, and below 0.25 little separation. These cutoffs are only heuristics: the score depends on the distance matrix and resulting cluster structure, and it cannot show that titles are genuinely equivalent.

Agglomerative and spectral assign every title. DBSCAN and HDBSCAN can mark titles as noise; for these methods, silhouette is calculated only on assigned titles. **Coverage** is the percentage of all titles assigned to a cluster. The results table reports the actual coverage for each method: full-assignment methods are 100%, while density methods can be lower when titles are noise. Read coverage alongside silhouette because a density method may score only a smaller subset, so scores across methods are not a simple ranking. There are no ground-truth labels to verify which groups are semantically correct.

The next table reports one fixed reference setting for each method, all applied to the same similarity matrix. These settings make the comparison reproducible; they are examples, not optimized winners.


In [2]:
def score_labels(labels, distances):
    labels = np.asarray(labels).copy()
    assigned = labels != -1
    assigned_labels = labels[assigned]
    n_clusters = len(np.unique(assigned_labels))
    silhouette = float("nan")
    if 2 <= n_clusters < int(assigned.sum()):
        assigned_distances = distances[np.ix_(assigned, assigned)]
        silhouette = float(
            silhouette_score(assigned_distances, assigned_labels, metric="precomputed")
        )
    return {
        "labels": labels,
        "clusters": n_clusters,
        "noise": int((~assigned).sum()),
        "coverage": float(assigned.mean()),
        "silhouette": silhouette,
    }


def mark_small_clusters_as_noise(labels, minimum_size=2):
    labels = np.asarray(labels).copy()
    for label in np.unique(labels):
        if label != -1 and np.count_nonzero(labels == label) < minimum_size:
            labels[labels == label] = -1
    return labels


# One fixed average-linkage threshold is tested here; this is not a threshold sweep.
agglo_labels = AgglomerativeClustering(
    n_clusters=None, metric="precomputed", linkage="average", distance_threshold=0.70
).fit_predict(D)
agglo_metrics = score_labels(agglo_labels, D)

# Density-based reference settings used in the results table and graph.
dbscan_labels = DBSCAN(eps=0.40, min_samples=2, metric="precomputed").fit_predict(D)
dbscan_labels = mark_small_clusters_as_noise(dbscan_labels, minimum_size=2)
dbscan_metrics = score_labels(dbscan_labels, D)
if not np.isfinite(dbscan_metrics["silhouette"]):
    raise ValueError("DBSCAN reference setting did not produce a valid silhouette")

hdbscan_labels = HDBSCAN(
    min_cluster_size=2,
    min_samples=2,
    metric="precomputed",
    copy=True,
).fit_predict(D)
hdbscan_metrics = score_labels(hdbscan_labels, D)
if not np.isfinite(hdbscan_metrics["silhouette"]):
    raise ValueError("HDBSCAN reference setting did not produce a valid silhouette")

# Spectral: keep 15 nearest neighbors; choose k from eigengaps in 20-80 (the separate notebook also swept k=10-100 by 5).
# Spectral clustering, following 5_spectral.ipynb.
K_NEIGHBORS = 15
W = S.copy()
np.fill_diagonal(W, 0)
thresholds = np.sort(W, axis=1)[:, -K_NEIGHBORS][:, None]
W = np.where((W >= thresholds) | (W >= thresholds.T), W, 0)
degree = W.sum(axis=1)
inverse_sqrt_degree = 1 / np.sqrt(np.maximum(degree, 1e-12))
laplacian = np.eye(n) - W * np.outer(inverse_sqrt_degree, inverse_sqrt_degree)
eigenvalues, eigenvectors = np.linalg.eigh(laplacian)
K_MIN, K_MAX = 20, 80
gaps = np.diff(eigenvalues[: K_MAX + 1])
k_spectral = int(np.argmax(gaps[K_MIN - 1 : K_MAX]) + K_MIN)
U = eigenvectors[:, :k_spectral]
U = U / np.maximum(np.linalg.norm(U, axis=1, keepdims=True), 1e-12)
spectral_labels = KMeans(n_clusters=k_spectral, n_init=10, random_state=0).fit_predict(U)
spectral_metrics = score_labels(spectral_labels, D)

clusterings = {
    "Agglomerative": {"labels": agglo_metrics["labels"], "metrics": agglo_metrics, "params": "average, threshold=0.70"},
    "DBSCAN": {"labels": dbscan_metrics["labels"], "metrics": dbscan_metrics, "params": "eps=0.40, min_samples=2"},
    "HDBSCAN": {"labels": hdbscan_metrics["labels"], "metrics": hdbscan_metrics, "params": "min_cluster_size=2, min_samples=2"},
    "Spectral": {"labels": spectral_metrics["labels"], "metrics": spectral_metrics, "params": f"k={k_spectral}, k_neighbors={K_NEIGHBORS}, seed=0"},
}

print("Cluster graph uses the fixed reference settings shown in the results table.")

Cluster graph uses the fixed reference settings shown in the results table.


In [3]:
print(f"{'Method':<16} {'Parameters':<48} {'Clusters':>8} {'Silhouette':>11} {'Coverage (%)':>13}")
print('-' * 100)
for name, result in clusterings.items():
    metrics = result["metrics"]
    print(f"{name:<16} {result['params']:<48} {metrics['clusters']:>8} {metrics['silhouette']:>11.4f} {metrics['coverage']:>12.1%}")


Method           Parameters                                       Clusters  Silhouette  Coverage (%)
----------------------------------------------------------------------------------------------------
Agglomerative    average, threshold=0.70                               138      0.1841       100.0%
DBSCAN           eps=0.40, min_samples=2                                59      0.4755        37.1%
HDBSCAN          min_cluster_size=2, min_samples=2                     106      0.2813        73.3%
Spectral         k=25, k_neighbors=15, seed=0                           25      0.1217       100.0%


### Results graphs

Each panel applies one method's labels to the same t-SNE projection. Titles report cluster count, silhouette, and coverage. The projection helps inspect grouping but can distort distances, so use the table's metrics—not visual spacing—to compare results.

In [4]:
embedding = TSNE(
    n_components=2,
    metric="precomputed",
    init="random",
    learning_rate="auto",
    perplexity=30,
    random_state=0,
).fit_transform(D)

figure = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=[
        f"{name}: {result['metrics']['clusters']} clusters; silhouette={result['metrics']['silhouette']:.3f}; coverage={result['metrics']['coverage']:.0%}"
        for name, result in clusterings.items()
    ],
)
for index, (name, result) in enumerate(clusterings.items()):
    row, col = index // 2 + 1, index % 2 + 1
    labels = result["labels"]
    noise = labels == -1
    assigned = ~noise
    figure.add_trace(
        go.Scattergl(
            x=embedding[assigned, 0],
            y=embedding[assigned, 1],
            mode="markers",
            marker={"color": labels[assigned], "colorscale": "Turbo", "size": 7, "opacity": 0.8, "showscale": False},
            text=np.asarray(attributes)[assigned],
            hovertemplate="%{text}<extra>cluster</extra>",
            showlegend=False,
        ),
        row=row,
        col=col,
    )
    if noise.any():
        figure.add_trace(
            go.Scattergl(
                x=embedding[noise, 0],
                y=embedding[noise, 1],
                mode="markers",
                marker={"color": "#999999", "size": 6, "symbol": "x", "opacity": 0.65},
                text=np.asarray(attributes)[noise],
                hovertemplate="%{text}<extra>noise</extra>",
                showlegend=False,
            ),
            row=row,
            col=col,
        )
figure.update_xaxes(title_text="t-SNE 1")
figure.update_yaxes(title_text="t-SNE 2")
figure.update_layout(template="simple_white", height=850, width=1100, title="Product-attribute clusters (shared distance projection)")
figure.show()

## Interpreting the results

The reference runs reveal a trade-off rather than a winner: DBSCAN's higher silhouette applies to the smallest assigned subset, while HDBSCAN retains more titles with a lower score. Agglomerative and spectral provide full coverage, but their scores suggest weak separation under this distance matrix.

These are internal measures of structure in the chosen distances, not a test of semantic correctness. Treat the cluster graph as a prompt for inspection and the scores as comparative evidence with the limitations described above.

## Useful literature

These papers were published in 2017 or later. They provide methodological background; their results do not directly validate this catalog or the current blend.

- [Sentence-BERT: Sentence Embeddings using Siamese BERT-Networks (2019)](https://aclanthology.org/D19-1410/) introduces sentence embeddings compared with cosine similarity, relevant to semantic text matching and clustering.
- [MTEB: Massive Text Embedding Benchmark (2023)](https://aclanthology.org/2023.eacl-main.148/) evaluates embeddings across tasks, including clustering, and finds no model is best for every task. It supports validating embeddings on the target task.
- [Qwen3 Embedding: Advancing Text Embedding and Reranking Through Foundation Models (2025)](https://arxiv.org/abs/2506.05176) describes the Qwen3 embedding family used here. It is useful for explaining the model, but its benchmark results do not establish performance on this catalog.
- [BEIR: A Heterogeneous Benchmark for Zero-shot Evaluation of Information Retrieval Models (2021)](https://arxiv.org/abs/2104.08663) compares lexical and neural retrieval across varied datasets, providing context for combining lexical and semantic signals.
- [Deep Entity Matching with Pre-Trained Language Models (2020; VLDB 2021)](https://arxiv.org/abs/2004.00584) studies matching records with pretrained language models. It is relevant to duplicate-feature matching and could inform a future supervised baseline if labeled pairs are collected.
- [Comparison of Internal Clustering Validation Indices for Prototype-Based Clustering (2017)](https://doi.org/10.3390/a10030105) compares internal validation indices and their behavior. Its prototype-based experiments offer background on interpreting internal scores, but do not directly validate this distance-matrix application.

For the exact scorer behavior, see the [RapidFuzz scorer documentation](https://rapidfuzz.github.io/RapidFuzz/Usage/fuzz.html); this is implementation documentation, not a research article.